# 分類性能と公平性の比較

通常 ResNet と attribute-invariant ResNet の best validation-AUROC checkpoint を同一 test split で比較する。

In [ ]:
# ruff: noqa: E402
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss, roc_auc_score

ANALYSIS_DIR = Path.cwd()
REPOSITORY_ROOT = ANALYSIS_DIR.parents[2]
sys.path.insert(0, str(REPOSITORY_ROOT))
from projects.hypernet_e2e.data.evaluation_attributes import add_age_groups
from projects.hypernet_e2e.utils.metrics import compute_fairness_metrics

CACHE_DIR = ANALYSIS_DIR / "cache"
RESULTS_DIR = ANALYSIS_DIR / "results"
SPLIT = "test"
RUNS = {
    "ResNet": "20260921T103036Z-resnet-chexpert-s42-5538",
    "attribute-invariant ResNet": "20260921T125711Z-resnet-chexpert-attribute-invariant-s42-9fd3",
}

In [ ]:
def load_cache(label, run_id):
    with np.load(CACHE_DIR / f"{run_id}_{SPLIT}.npz", allow_pickle=False) as cached:
        result = {key: cached[key] for key in cached.files}
    result["label"] = label
    result["metadata"] = json.loads(str(result["metadata"]))
    return result


caches = [load_cache(label, run_id) for label, run_id in RUNS.items()]

In [ ]:
def classification_metrics(cached):
    target, probabilities = cached["target"], cached["probabilities"]
    return {
        "model": cached["label"],
        "split": SPLIT,
        "run_id": cached["metadata"]["run_id"],
        "n_examples": len(target),
        "accuracy": accuracy_score(target, cached["predictions"]),
        "balanced_accuracy": balanced_accuracy_score(target, cached["predictions"]),
        "auroc": roc_auc_score(target, probabilities[:, 1]),
        "cross_entropy": log_loss(target, probabilities, labels=np.arange(probabilities.shape[1])),
    }


comparison = pd.DataFrame(classification_metrics(cached) for cached in caches)
RESULTS_DIR.mkdir(exist_ok=True)
comparison.to_csv(RESULTS_DIR / f"classification_performance_{SPLIT}.csv", index=False)
comparison

In [ ]:
FAIRNESS_COLUMNS = ["sex", "race", "ethnicity", "age_group_65"]
split_frame = pd.read_csv(REPOSITORY_ROOT / "data" / "chexpert" / "splits" / f"{SPLIT}.csv")
split_frame = add_age_groups(split_frame, {"age_group_65": {"source": "age", "boundaries": [65]}})
attributes = {
    "categorical": torch.as_tensor(split_frame[FAIRNESS_COLUMNS].to_numpy(), dtype=torch.long),
    "categorical_missing": torch.as_tensor(
        split_frame[[f"{name}_missing" for name in FAIRNESS_COLUMNS]].to_numpy(dtype=bool),
        dtype=torch.bool,
    ),
}


def fairness_rows(cached):
    metrics = compute_fairness_metrics(
        torch.from_numpy(cached["logits"]),
        torch.from_numpy(cached["target"]),
        attributes,
        {"categorical": FAIRNESS_COLUMNS},
    )
    return [{"model": cached["label"], "split": SPLIT, "attribute": name, **values} for name, values in metrics.items()]


fairness = pd.DataFrame(row for cached in caches for row in fairness_rows(cached))
fairness.to_csv(RESULTS_DIR / f"fairness_metrics_{SPLIT}.csv", index=False)
fairness

この比較は seed 42 の各1 run に限る。公平性の差を主張する前に、複数 seed で再検証する。